# 10. Human-in-the-Loop Approval Workflow
**Industry:** Financial Services

Build a LangGraph workflow that pauses execution and waits for human approval before proceeding with a sensitive action (e.g., executing a trade).

In [1]:
!pip install langgraph langchain langchain-openai


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class TradeState(TypedDict):
    trade_details: str
    approved: bool

def prep_trade(state: TradeState):
    print(f"\n[System] Prepping trade: {state['trade_details']}")
    return state

def execute_trade(state: TradeState):
    if state.get("approved"):
        print("\n[System] Trade Executed successfully!")
    else:
        print("\n[System] Trade was REJECTED.")
    return state

workflow = StateGraph(TradeState)
workflow.add_node("prep_trade", prep_trade)
workflow.add_node("execute_trade", execute_trade)

workflow.add_edge(START, "prep_trade")
workflow.add_edge("prep_trade", "execute_trade")
workflow.add_edge("execute_trade", END)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory, interrupt_before=["execute_trade"])

config = {"configurable": {"thread_id": "trade_1"}}
state = {"trade_details": "Buy 100 shares of AAPL", "approved": False}

# Run until interrupt
print("Starting workflow...")
app.invoke(state, config=config)

# User provides approval
print("\n--- WAITING FOR HUMAN APPROVAL ---")
user_input = "yes"

# Update state and resume
app.update_state(config, {"approved": user_input.lower() == "yes"})
print("Resuming workflow...")
app.invoke(None, config=config)

Starting workflow...

[System] Prepping trade: Buy 100 shares of AAPL

--- WAITING FOR HUMAN APPROVAL ---
Resuming workflow...

[System] Trade Executed successfully!


{'trade_details': 'Buy 100 shares of AAPL', 'approved': True}